## Compare Profiles Between IGs

This uses the validators compare function to compare a set of profiles:

- inputs a user supplied csv table of tuples of the profiles to compare
 
- output are folders with the comparisons

---
### Instructions

- download the latest version of the validator from
https://github.com/hapifhir/org.hl7.fhir.core/releases/latest/download/validator_cli.jar
Comparing Profiles
   - The validator can compare profiles. To compare profiles, use the following
parameters:

   ~~~
   java -jar validator_cli.jar -compare -dest /home/user/work/ig-comparison
   -version 4.0
   -ig hl7.fhir.us.carin-bb#1.1.0
   -ig hl7.fhir.us.davinci-crd#1.0.0
   -left http://hl7.org/fhir/us/carin-bb/StructureDefinition/C4BB-Patient
   -right http://hl7.org/fhir/us/davinci-crd/StructureDefinition/profile-patient
   ~~~

   **Parameters Explanation:**

   - -compare: tell the validator to rin the comparison logic
   - -dest: folder to produce the output. This must exist, and the validator will
   overwrite existing content if it needs to. The output isnt simple - see below
   - -version Maj.Min - the version to use. You can leave this out and let the
   validator infer this, but its there so that you can compare profiles across
   versions. E.g. if you specify -version 4.0, the profiles will both be treated as
   R4 profiles, even if they arent
   - -ig - a repeating parameter that specifies the packages to load, that contain
   the profiles you want to compare
   - -left and -right - the two profiles to compare. Theres no functional difference
   between left and right, except that the comparison will keep to left and right
   consistently

### Output

- The output starts at index.html. The output isnt simple (and may take some time
to generate)

- For each pair of profiles, the comparison generates a union and an
intersections. The union is that total set of things that are allowed by either
profile - thats what you could expect to read as a consumer of resources
conforming to both profiles. The intersection is the set of things that both
implementation guides allow - this is what you are required/allowed to write
into a resource if you are creating one that must conform to both profiles. Its
possible that the intersection will be empty - theres no valid instance that
can conform to both profiles. In that case, consult the IG authors.

- The comparison will (must) compare the sub-profiles that the profiles refer to.
E.g. if both profiles on a resource such as MedicationAdministration refer to a
profile of patient, those profiles will also be compared.

---


In [ ]:

from pandas import read_csv
from pandas import DataFrame as df
from pathlib import Path
import shutil
import re
import jinja2
from IPython.display import display as Display, HTML

# =======these globals change for each comparison =======
path='/Users/ehaas/Downloads/validator_cli.jar' # path to publisher
# out_path='/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/utils/out/profile_compare' # path to the root destination folder for individual compare folders
OUT=f'docs'
VERSION='4.0.1' # version of FHIR to use

# IG1='hl7.fhir.us.davinci-pas#2.0.0' # path to PAS CI build
# IG2='hl7.fhir.us.davinci-cdex#2.0.0 ' # path to path to CDex 2.0.0

IG1='fhir.argonaut.r2#1.0.0' # path to IG Package 1 = ARGONAUT DQ
IG2='hl7.fhir.us.core#dev' # path to the latest IG Package
IG2_VERSION = '10.0.0'


# IG1='hl7.fhir.uv.ipa#current' # path to IPA "hl7.fhir.uv.ipa","version" : "1.0.0-preview
# IG2='hl7.fhir.us.core#current' # path to IG Package 2 = US Core 6.0.0

LEFT =  'ARGONAUT DQ'
# LEFT =  'IPA'
# LEFT = 'PAS'

RIGHT = 'US Core 9.0.0'
# RIGHT = 'CDEX'
# ======================================================

# =======these globals do not change for each comparison =======
target_folder = 'comparison-argo' #'comparison-ipa' # name of the folder to create in the out_path
# target_folder = 'comparison-ipa' # name of the folder to create in the out_path
# target_folder = 'comparison-pas' # name of the folder to create in the out_path
# ======================================================
OUT

'docs'

### Fetch Profiles to compare from CSV table

In [7]:
# =======these map  change for each comparison =======
profile_map = '/Users/ehaas/Documents/FHIR/US-Core/input/images-source/ArgoDQ-USCore-ProfileMap.csv'
# profile_map = '/Users/ehaas/Documents/FHIR/fhir-ipa/input/images/source/IPA-USCore-ProfileMap.csv'
# profile_map = 'in/PAS-CDEX-ProfileMap.csv'
# ======================================================
df = read_csv(profile_map,na_filter=False)

df

,ArgoDQ,USCore9_0,USCore9_1
0,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
1,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
2,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
3,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
4,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,http://hl7.org/fhir/us/core/StructureDefinitio...
5,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
6,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,http://hl7.org/fhir/us/core/StructureDefinitio...
7,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
8,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,
9,http://fhir.org/guides/argonaut/StructureDefin...,http://hl7.org/fhir/us/core/StructureDefinitio...,


### run the validator to compare profiles

~~~
java -Xmx4G -jar $path -compare -dest $OUT -version $VERSION -ig $IG1 -ig $IG2 -left $LEFT -right $RIGHT
~~~

### passing variables between bash and python

<!-- ~~~
a=10
b=20
~~~

~~~
%%bash -s $a $b --out c
echo "this is displayed"
echo "the last thing printed to stdout will be passed back in the variable c"
echo "a + b = $(($1+$2))"
~~~~

~~~
print(c)
~~~
this is displayed
the last thing printed to stdout will be passed back in the variable c
a + b = 30 -->

In [8]:
def compare(path, OUT, VERSION, IG1, IG2, LEFT, RIGHT):
# %%bash -s $path $OUT $VERSION $IG1 $IG2 $LEFT $RIGHT
# echo $1 $2 $3 $4 $5 $6 $7
  print(f'OUT: {OUT}')
  !mkdir -p {OUT}
  !java -Xmx4G -jar {path} -compare -dest {OUT} -version {VERSION} -ig {IG1} -ig {IG2} -left {LEFT} -right {RIGHT}
  print('============== on exit the output folder will open up your browser================')
  return


### clean up any old files


In [9]:
!ls {OUT}/*.*
!rm {OUT}/*.*


docs/administration-module-person.png
docs/administration.jpg
docs/clinicalreasoning-components-diagram.jpg
docs/fhr.icns
docs/icon-administration.png
docs/icon-clinical.png
docs/icon-documentation.png
docs/icon-fhir-720.png
docs/icon-implementation.png
docs/icon-infrastructure.png
docs/index.html
docs/sd+1-argo-allergyintolerance-us-core-allergyintolerance-intersection.html
docs/sd+1-argo-allergyintolerance-us-core-allergyintolerance-intersection.json
docs/sd+1-argo-allergyintolerance-us-core-allergyintolerance-union.html
docs/sd+1-argo-allergyintolerance-us-core-allergyintolerance-union.json
docs/sd+1-argo-allergyintolerance-us-core-allergyintolerance.html
docs/sd+1-argo-birthsex-us-core-birthsex-intersection.html
docs/sd+1-argo-birthsex-us-core-birthsex-intersection.json
docs/sd+1-argo-birthsex-us-core-birthsex-union.html
docs/sd+1-argo-birthsex-us-core-birthsex-union.json
docs/sd+1-argo-birthsex-us-core-birthsex.html
docs/sd+1-argo-careplan-us-core-careplan-intersection.html
docs/s

### Iterate over the Rows and perform comparison using the validator

- some rows have 1 -> many profiles to compare
- some rows have no profile to compare 1-> 0  and 0 -> 1
- saves to the Docs folder  = current 

In [10]:


for r in df.itertuples():
    print(r.Index, r[1], r[2])
    if r[1] and r[2]:
        compare(path, OUT, VERSION, IG1, IG2, r[1], r[2])
    if r[3]:
        RFILENAME=r[3].split('/')[-1]
        # OUT=f'{out_path}/temp_out'
        compare(path, OUT, VERSION, IG1, IG2, r[1], r[3])
        # use bash to move the files to the r[2] place
        # !cp {OUT}/*.html {out_path}
        # !cp {OUT}/*.json {out_path}
    # if r.Index == 2:
    #     break
    # pull out *json and *html files from the output folder and put them in the first folder with the name of the profile
    # combine the html tables into a single html file


0 http://fhir.org/guides/argonaut/StructureDefinition/argo-allergyintolerance http://hl7.org/fhir/us/core/StructureDefinition/us-core-allergyintolerance
OUT: docs
FHIR Validation tool Version 6.9.6 (Git# 3773c292b3f4). Built 2026-04-11T09:02:03.303Z (192 days old)
The option '-compare' is no longer in use. Using 'compare' 
  Java:   17.0.2 from /Library/Java/JavaVirtualMachines/jdk-17.0.2.jdk/Contents/Home on aarch64 (64bit). 4096MB available
  Paths:  Current = /Users/ehaas/Documents/FHIR/Data-Query-US-Core-Comparison, Package Cache = /Users/ehaas/.fhir/packages
  Params: compare -dest docs -version 4.0.1 -ig fhir.argonaut.r2#1.0.0 -ig hl7.fhir.us.core#dev -left http://fhir.org/guides/argonaut/StructureDefinition/argo-allergyintolerance -right http://hl7.org/fhir/us/core/StructureDefinition/us-core-allergyintolerance
Loading
  Loading FHIR v4.0.1 from hl7.fhir.r4.core#4.0.1
  Load hl7.terminology.r4#6.2.0 - 4288 resources (00:05.800)
  Load hl7.fhir.uv.extensions.r4#5.2.0 - 759 resour

### Remove unwanted output files

- union and intersection files not used for now.

In [11]:
!rm {OUT}/template-*.html
!rm {OUT}/redirect.*.*
!rm {OUT}/*-intersection.html
!rm {OUT}/*-intersection.json
!rm {OUT}/*-union.html
!rm {OUT}/*-union.json
!rm {OUT}/xx+*.html


zsh:1: no matches found: docs/template-*.html
zsh:1: no matches found: docs/redirect.*.*


### Rename output files to remove `+N` suffix

The validator generates filenames like `sd+1-...` and `vs+8-...`. Strip the `+N` so they become `sd-...` and `vs-...`.

In [ ]:
out_dir = Path(OUT)
for f in out_dir.iterdir():
    new_name = re.sub(r'\+\d+', '', f.name)
    if new_name != f.name:
        f.rename(out_dir / new_name)
        print(f"Renamed: {f.name} -> {new_name}")

Renamed: vs+8-observation-codes-us-core-laboratory-test-codes.html -> vs-observation-codes-us-core-laboratory-test-codes.html
Renamed: vs+9-observation-methods-observation-methods.html -> vs-observation-methods-observation-methods.html
Renamed: vs+6-c80-facilitycodes-c80-facilitycodes.html -> vs-c80-facilitycodes-c80-facilitycodes.html
Renamed: sd+1-argo-diagnosticreport-us-core-diagnosticreport-lab.html -> sd-argo-diagnosticreport-us-core-diagnosticreport-lab.html
Renamed: vs+5-formatcodes-v3-HL7FormatCodes.html -> vs-formatcodes-v3-HL7FormatCodes.html
Renamed: vs+4-goal-start-event-goal-start-event.html -> vs-goal-start-event-goal-start-event.html
Renamed: vs+9-body-site-body-site.html -> vs-body-site-body-site.html
Renamed: vs+5-manifestation-or-symptom-manifestation-or-symptom.html -> vs-manifestation-or-symptom-manifestation-or-symptom.html
Renamed: vs+3-report-codes-us-core-laboratory-test-codes.html -> vs-report-codes-us-core-laboratory-test-codes.html
Renamed: vs+4-procedure-ca

### Copy missing PNG assets into temp_out

The validator output references PNG files (tree connectors, icons, etc.) that it doesn't generate itself. Copy them from the existing US-Core comparison-argo images folder.

In [ ]:
# PNG assets referenced by the validator's HTML output but not generated by it
ASSET_SOURCE = Path('/Users/ehaas/Documents/FHIR/US-Core/input/images/comparison-argo')
MISSING_PNGS = [
    'help16.png',
    'icon_extension_simple.png',
    'icon_modifier_extension_simple.png',
    'icon_resource.png',
    'tbl_bck0.png',
    'tbl_blank.png',
    'tbl_spacer.png',
    'tbl_vjoin.png',
    'tbl_vjoin_end.png',
    'tbl_vline.png',
]

dest = Path(OUT)
copied = []
missing = []
for png in MISSING_PNGS:
    src = ASSET_SOURCE / png
    if src.exists():
        shutil.copy2(src, dest / png)
        copied.append(png)
    else:
        missing.append(png)

print(f"Copied {len(copied)} PNG assets to {dest}")
if missing:
    print(f"WARNING: {len(missing)} files not found in source: {missing}")

Copied 10 PNG assets to docs


### Create own index.html from the table

In [ ]:
profcomp = Path(r'/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/utils/out/profile_compare/profcomp')
env = jinja2.Environment(loader=jinja2.FileSystemLoader(searchpath='.'), trim_blocks=True, lstrip_blocks=True)
template = env.get_template('profcompare.j2',)

# temporary override globals for testing and fixing US Core
# profile_map = '/Users/ehaas/Documents/FHIR/US-Core/input/images-source/ArgoDQ-USCore-ProfileMap.csv'
# df = read_csv(profile_map,na_filter=False)

# LEFT =  'ARGONAUT DQ'
RIGHT = 'US Core 9.0.0'

# OUT= '/Users/ehaas/Documents/FHIR/US-Core/input/images/comparison-argo'

# code that fills in display_dictionary with the values to send to the template
sd_complist = [f.name for f in Path(OUT).glob('sd-*.html')] # check if compare.html exists

# get vs-*.html files
# inspect the title of the html file to get the name of the value set
# create a Dictionary of the value set name and the html file
# create a template that will create a table of contents

vs_dict = {f.name:re.findall("<title>(.*?)</title>", f.read_text(),)[0] for f in Path(OUT).glob('vs-*.html')}

my_index = template.render(LEFT=LEFT,
                            RIGHT=RIGHT,
                            profcomp=df.itertuples(),
                            sd_complist=sd_complist,
                            vs_dict=vs_dict)
Display(HTML(my_index))
# index_out = Path(r'/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/utils/out/profile_compare/profcomp/index.html')
# index_out.write_text(my_index)
out = Path(f'{OUT}/index.html')
out.write_text(my_index)
sd_complist

['sd-argo-diagnosticreport-us-core-diagnosticreport-note.html',
 'sd-argo-procedure-us-core-procedure.html',
 'sd-argo-goal-us-core-goal.html',
 'sd-argo-medication-us-core-medication.html',
 'sd-argo-allergyintolerance-us-core-allergyintolerance.html',
 'sd-argo-condition-us-core-condition-encounter-diagnosis.html',
 'sd-argo-documentreference-us-core-documentreference.html',
 'sd-argo-birthsex-us-core-birthsex.html',
 'sd-argo-immunization-us-core-immunization.html',
 'sd-argo-diagnosticreport-us-core-diagnosticreport-lab.html',
 'sd-argo-condition-us-core-condition-problems-health-concerns.html',
 'sd-argo-careplan-us-core-careplan.html']

### Copy output to Version Folder

using bash

In [ ]:
version_subdir = Path() / OUT /  IG2_VERSION
version_subdir.mkdir(exist_ok=True)
# print(f'OUT = {OUT}')
# print(f'version_subdir= {version_subdir}')
for file in Path(OUT).iterdir():
    if file.is_file():
        shutil.copy2(file, version_subdir)

###  NOT USED --- Create a list of file to exempt from headers,etc ---

add output to sushi-config.yaml file as a "html-exempt" file


### NOT USES ---- Clean up Compare File

Clean up artifacts in an HTML file:

1. Replace ` null"` with `"`
2. Strip <a href="??...">text</a> down to just `text`

1. look for "null" and remove  ( e.g. in /Users/ehaas/Documents/Python/Jupyter/MyNotebooks/utils/out/profile_compare/temp_out/sd-argo-goal-us-core-goal.html)
   1. find ` null"` and replace with `"`
2. fix bad links for publisher using regex
   1. find '<a href="\?\?.*?>(.*?)</a>'
   2. replace '$1'